# Practice - full flow on gym_members.csv

A gym wants to know which members will renew their membership.

Do all the steps we did in class, on this new file. We want to predict `renewed`.

**Expected accuracy: about 0.82**

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score

data_path = Path("../..") / "Data" / "gym_members.csv"
df = pd.read_csv(data_path)
df.head()

,member_id,age,plan_type,workouts_per_week,session_minutes,distance_km,months_active,trainer,referral_source,renewed
0,GYM3230,43.0,Pro,3,56.0,9.3,35.0,No,Walk-in,No
1,GYM3391,16.0,Basic,3,NaN,10.6,17.0,No,Friend,No
2,GYM3137,16.0,Basic,4,61.0,5.2,2.0,No,Instagram,No
3,GYM3339,28.0,Basic,5,NaN,2.5,31.0,Yes,Walk-in,Yes
4,GYM3291,32.0,Pro,6,NaN,3.3,15.0,No,Instagram,Yes


---
## Step 1 - Load and look

Expected shape: `(500, 10)`

In [2]:
print(f"The dataset has {df.shape[0]} rows and {df.shape[1]} columns.")
print("\nColumn and data-type summary:")
df.info()
print("\nMissing values in each column:")
print(df.isna().sum())

The dataset has 500 rows and 10 columns.

Column and data-type summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   member_id          500 non-null    object 
 1   age                500 non-null    float64
 2   plan_type          500 non-null    object 
 3   workouts_per_week  500 non-null    int64  
 4   session_minutes    442 non-null    float64
 5   distance_km        479 non-null    float64
 6   months_active      491 non-null    float64
 7   trainer            500 non-null    object 
 8   referral_source    500 non-null    object 
 9   renewed            500 non-null    object 
dtypes: float64(4), int64(1), object(5)
memory usage: 39.2+ KB

Missing values in each column:
member_id             0
age                   0
plan_type             0
workouts_per_week     0
session_minutes      58
distance_km          21
months_a

In [3]:
print("Missing values in each column:")
print(df.isna().sum())
print("\nRenewal counts:")
print(df["renewed"].value_counts())

Missing values in each column:
member_id             0
age                   0
plan_type             0
workouts_per_week     0
session_minutes      58
distance_km          21
months_active         9
trainer               0
referral_source       0
renewed               0
dtype: int64

Renewal counts:
renewed
Yes    270
No     230
Name: count, dtype: int64


---
## Step 2 - Separate numeric and categorical columns

Expected: 5 numeric, 3 categorical. `member_id` and `renewed` go in neither list.

In [4]:
numeric_columns = [
    "age",
    "workouts_per_week",
    "session_minutes",
    "distance_km",
    "months_active",
]
categorical_columns = ["plan_type", "trainer", "referral_source"]

print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

Numeric columns: ['age', 'workouts_per_week', 'session_minutes', 'distance_km', 'months_active']
Categorical columns: ['plan_type', 'trainer', 'referral_source']


---
## Step 3 - Split into X and y

Expected: X `(500, 8)`, y `(500,)`

In [5]:
X = df[numeric_columns + categorical_columns].copy()
y = df["renewed"].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

X shape: (500, 8)
y shape: (500,)


,age,workouts_per_week,session_minutes,distance_km,months_active,plan_type,trainer,referral_source
0,43.0,3,56.0,9.3,35.0,Pro,No,Walk-in
1,16.0,3,NaN,10.6,17.0,Basic,No,Friend
2,16.0,4,61.0,5.2,2.0,Basic,No,Instagram
3,28.0,5,NaN,2.5,31.0,Basic,Yes,Walk-in
4,32.0,6,NaN,3.3,15.0,Pro,No,Instagram


---
## Step 4 - Handle missing values

Fill every numeric column with its own mean.

Expected before: `session_minutes` 58, `distance_km` 21, `months_active` 9. After: all zero.

In [6]:
missing_before = X[numeric_columns].isna().sum()
print("Missing numeric values before filling:")
print(missing_before[missing_before > 0])

Missing numeric values before filling:
session_minutes    58
distance_km        21
months_active       9
dtype: int64


In [7]:
for column in numeric_columns:
    X[column] = X[column].fillna(X[column].mean())

print("Missing values after filling:")
print(X.isna().sum())

Missing values after filling:
age                  0
workouts_per_week    0
session_minutes      0
distance_km          0
months_active        0
plan_type            0
trainer              0
referral_source      0
dtype: int64


---
## Step 5 - Encode and scale together

Use one `ColumnTransformer`.

Expected shape: `(500, 14)`

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_columns),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
    ]
)

In [9]:
X_transformed = preprocessor.fit_transform(X)
print(f"Transformed feature shape: {X_transformed.shape}")
print("The five numeric columns are scaled, and the three text columns are one-hot encoded.")

Transformed feature shape: (500, 14)
The five numeric columns are scaled, and the three text columns are one-hot encoded.


---
## Step 6 - Train and test split

Expected: train `(400, 14)`, test `(100, 14)`

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training data: {X_train.shape}")
print(f"Testing data: {X_test.shape}")

Training data: (400, 14)
Testing data: (100, 14)


---
## Step 7 - Train the model

`KNeighborsClassifier` with `n_neighbors=5`, then predict on the test set.

In [11]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [12]:
predictions = knn_model.predict(X_test)
print("First ten predictions:", predictions[:10])

First ten predictions: ['Yes' 'Yes' 'Yes' 'No' 'No' 'Yes' 'No' 'Yes' 'Yes' 'Yes']


---
## Step 8 - Score

Print the accuracy of your model.

In [13]:
accuracy = accuracy_score(y_test, predictions)
print(f"Model accuracy: {accuracy:.2%}")

Model accuracy: 81.00%


---
## Step 9 - Predict for one new member

Build a one row DataFrame with your own values and predict whether that member renews.

Use `transform`, not `fit_transform`.

In [14]:
new_member = pd.DataFrame(
    {
        "age": [27],
        "workouts_per_week": [4],
        "session_minutes": [60],
        "distance_km": [5.5],
        "months_active": [12],
        "plan_type": ["Pro"],
        "trainer": ["Yes"],
        "referral_source": ["Friend"],
    }
)

# Use transform because the preprocessor was already fitted on the gym data.
new_member_transformed = preprocessor.transform(new_member)
new_prediction = knn_model.predict(new_member_transformed)[0]
print(f"Prediction for the new member: {new_prediction}")
print("Interpretation: the model expects this member to renew." if new_prediction == "Yes" else "Interpretation: the model does not expect this member to renew.")

Prediction for the new member: Yes
Interpretation: the model expects this member to renew.
